# VLM Mathematical Reasoning Benchmark: GSM8K Multi-Modal Evaluation
## Model: LLaVA-1.6 / LLaVA-NeXT 7B (`llava-hf/llava-v1.6-mistral-7b-hf`)

This notebook evaluates **LLaVA-1.6 (LLaVA-NeXT) Mistral-7B** on the first 100 GSM8K problems under four conditions:

| Condition | Input | Vision | Description |
|-----------|-------|--------|-------------|
| **1 — Baseline** | Pure text | ❌ | Standard language reasoning |
| **2 — Rendered Image** | Clean PNG | ✅ | Digitally rendered text image |
| **3 — Screenshot** *(optional)* | Browser screenshot | ✅ | Real-world visual noise |
| **4 — Mismatch** | Image of problem *i*, text of *i+1* | ✅ | Which modality does the model trust? |

### Why LLaVA-1.6?
- ✅ Fits on a **T4 GPU** (16 GB) via 4-bit quantisation
- ✅ **Significantly better OCR** than LLaVA-1.5 via high-resolution tiling (up to 672×672 per tile)
- ✅ **Mistral-7B backbone** improves reasoning quality over Vicuna-7B
- ✅ LLaVA-NeXT architecture with proper `LlavaNextProcessor`

> **Runtime:** Set to **GPU (T4)** → `Runtime > Change runtime type > T4 GPU`

## 1. Install & Import Required Libraries

LLaVA-1.6 requires `transformers >= 4.39` and `bitsandbytes` for 4-bit quantisation on the T4.

In [ ]:
# Core ML / VLM stack
!pip install -q "transformers>=4.39.0" accelerate bitsandbytes
# Dataset + evaluation
!pip install -q datasets evaluate
# Utilities
!pip install -q tqdm pillow
# Optional: Selenium for Condition 3 (screenshot)
!pip install -q selenium webdriver-manager

In [ ]:
import transformers
print(f"transformers version: {transformers.__version__}")
assert tuple(int(x) for x in transformers.__version__.split('.')[:2]) >= (4, 39), \
    "Upgrade: pip install -q 'transformers>=4.39.0'"

In [ ]:
import os, re, textwrap
from collections import Counter

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
from datasets import load_dataset

# LLaVA-NeXT specific imports
from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    BitsAndBytesConfig,
)

torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# LLaVA-1.6 / LLaVA-NeXT Mistral-7B
# Alternative backbone: "llava-hf/llava-v1.6-vicuna-7b-hf"
MODEL_NAME = "llava-hf/llava-v1.6-mistral-7b-hf"

USE_4BIT = True   # Recommended for T4 — saves ~8 GB VRAM

DATASET_NAME   = "gsm8k"
DATASET_CONFIG = "main"
NUM_PROBLEMS   = 100
MAX_NEW_TOKENS = 512   # LLaVA-1.6 CoT chains benefit from more room

# 672px matches LLaVA-NeXT's native tile resolution for best OCR
IMG_WIDTH      = 672
IMG_FONT_SIZE  = 22
IMG_PADDING    = 40
IMG_BG_COLOR   = "white"
IMG_TEXT_COLOR = "black"

RUN_SCREENSHOT_CONDITION = False  # Set True to run Condition 3

OUTPUT_DIR = "vlm_benchmark_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/rendered_images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/screenshots", exist_ok=True)

print(f"Model         : {MODEL_NAME}")
print(f"4-bit quant   : {USE_4BIT}")
print(f"Problems      : {NUM_PROBLEMS}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Image width   : {IMG_WIDTH}px")
print(f"Screenshot run: {RUN_SCREENSHOT_CONDITION}")

## 3. Load GSM8K Dataset (First 100 Problems)

In [ ]:
full_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")
dataset      = full_dataset.select(range(NUM_PROBLEMS))

questions  = dataset["question"]
references = dataset["answer"]

print(f"Loaded {len(questions)} problems from GSM8K test split.")
print("\nSample:")
print("  Q:", questions[0][:120], "...")
print("  A:", references[0][-60:])

## 4. Load LLaVA-1.6 (Processor + Model)

`LlavaNextProcessor` handles LLaVA-NeXT's **high-resolution tiling** (up to 4 × 336×336 tiles), which is the key OCR improvement over LLaVA-1.5.
We use `BitsAndBytesConfig` (4-bit NF4) to fit the 7B model on a T4 GPU.

> For Condition 1 (text-only) we omit the `images=` argument — the vision pathway is simply unused.

In [ ]:
print(f"Loading processor: {MODEL_NAME} ...")
processor = LlavaNextProcessor.from_pretrained(MODEL_NAME)
print("Processor loaded.")
print(f"  Tokeniser : {type(processor.tokenizer).__name__}")
print(f"  Image proc: {type(processor.image_processor).__name__}")

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("First run downloads ~14 GB weights — may take 3-5 minutes.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("Model loaded.")
if USE_4BIT and DEVICE == "cuda":
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. Image Rendering Utility (Conditions 2 & 4)

Problems are rendered at 672px wide — matching LLaVA-NeXT's native tile resolution — on a white background for maximum OCR accuracy.

In [ ]:
def render_text_to_image(
    text: str,
    width: int  = IMG_WIDTH,
    font_size: int = IMG_FONT_SIZE,
    padding: int = IMG_PADDING,
    bg_color: str = IMG_BG_COLOR,
    text_color: str = IMG_TEXT_COLOR,
) -> Image.Image:
    """
    Render a math problem string into a clean PNG image.

    The text is word-wrapped to fit the specified width, rendered on a
    white background with a consistent monospace-style font. The canvas
    height is computed dynamically so the full problem always fits.

    Parameters
    ----------
    text      : The problem string to render.
    width     : Canvas width in pixels.
    font_size : Font size in points.
    padding   : Pixel padding on all four sides.
    bg_color  : Background colour string (e.g. 'white').
    text_color: Text colour string (e.g. 'black').

    Returns
    -------
    PIL.Image.Image
    """
    # Attempt to load a system font; fall back to default if unavailable
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except OSError:
        font = ImageFont.load_default()

    # Estimate characters per line based on pixel width and font size
    chars_per_line = max(20, (width - 2 * padding) // (font_size * 0.6))
    wrapped = textwrap.fill(text, width=int(chars_per_line))
    lines   = wrapped.split("\n")

    # Calculate canvas height from number of lines
    line_height = font_size + 8
    height      = 2 * padding + len(lines) * line_height + 10

    # Draw
    img  = Image.new("RGB", (width, height), color=bg_color)
    draw = ImageDraw.Draw(img)
    y    = padding
    for line in lines:
        draw.text((padding, y), line, fill=text_color, font=font)
        y += line_height

    return img


# ── Verify rendering with one example ────────────────────────────────────────
sample_img = render_text_to_image(questions[0])
sample_img.save(f"{OUTPUT_DIR}/rendered_images/sample_q0.png")
print(f"Sample rendered image saved: {OUTPUT_DIR}/rendered_images/sample_q0.png")
print(f"Image size: {sample_img.size[0]}×{sample_img.size[1]} px")
sample_img  # Display inline in Jupyter

## 6. Inference Functions

LLaVA-1.6 Mistral uses the **Mistral instruct prompt format**:
- Text-only: `[INST] {text} [/INST]`
- With image: `[INST] <image>\n{text} [/INST]`

The `<image>` token is replaced by the tiled patch embeddings by `LlavaNextProcessor`.

In [ ]:
def generate_text_only(question: str) -> str:
    """Condition 1: text-only, vision encoder unused."""
    prompt = (
        f"[INST] Solve the following math problem step by step. "
        f"Show your reasoning, then end with '#### <answer>'.\n\n"
        f"Problem: {question} [/INST]"
    )
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    n = inputs["input_ids"].shape[1]
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()


def generate_with_image(image: Image.Image) -> str:
    """Conditions 2 & 3: image-based inference via LLaVA-NeXT high-res tiling."""
    prompt = (
        "[INST] <image>\n"
        "The image shows a math word problem. Read it carefully and solve it "
        "step by step. End with '#### <answer>'. [/INST]"
    )
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    n = inputs["input_ids"].shape[1]
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()


print("Inference functions defined.")
print("  generate_text_only()  -> Condition 1")
print("  generate_with_image() -> Conditions 2, 3")

## 7. Answer Extraction & Evaluation Utilities

In [ ]:
def extract_numeric_answer(text: str):
    """Extract final numeric answer. Prefers '#### N' format, falls back to last number."""
    if not text or not text.strip():
        return None
    canon = re.search(r"####\s*([\d,\.]+)", text)
    if canon:
        try:
            return float(canon.group(1).replace(",", ""))
        except ValueError:
            pass
    numbers = [n for n in re.findall(r"[\d,]+\.?\d*", text)
               if n.replace(",", "").replace(".", "")]
    if numbers:
        try:
            return float(numbers[-1].replace(",", ""))
        except ValueError:
            pass
    return None


def answers_match(pred: str, ref: str) -> bool:
    p, r = extract_numeric_answer(pred), extract_numeric_answer(ref)
    if p is None or r is None:
        return False
    return round(p) == round(r)


def classify_error(prediction: str, reference: str) -> str:
    """Heuristic error type: correct / no_number / vision_error / arithmetic_error / reasoning_error."""
    if answers_match(prediction, reference):
        return "correct"
    if extract_numeric_answer(prediction) is None:
        return "no_number"
    vision_kw = ["cannot read", "can't read", "image is unclear",
                 "unable to see", "blurry", "illegible", "not visible",
                 "cannot see the text", "cannot identify"]
    if any(kw in prediction.lower() for kw in vision_kw):
        return "vision_error"
    return "arithmetic_error" if re.search(r"[\+\-\*\/\=]", prediction) else "reasoning_error"


def compute_accuracy(flags):
    return sum(flags) / len(flags) if flags else 0.0


print("Evaluation utilities defined.")

## 8. Condition 1 — Baseline: Pure Text (Vision Disabled)

No image is passed. This measures the Mistral-7B backbone's raw reasoning ability.

In [ ]:
print("=" * 60)
print("CONDITION 1: Text-Only Baseline")
print("=" * 60)

text_predictions, text_correct, text_errors = [], [], []

for q, ref in tqdm(zip(questions, references), total=NUM_PROBLEMS, desc="Cond 1"):
    pred = generate_text_only(q)
    text_predictions.append(pred)
    text_correct.append(answers_match(pred, ref))
    text_errors.append(classify_error(pred, ref))

acc_text = compute_accuracy(text_correct)
print(f"\nCondition 1 Accuracy: {acc_text:.2%}  ({sum(text_correct)}/{NUM_PROBLEMS})")

## 9. Condition 2 — Rendered Image (Vision Enabled, Clean Format)

Each problem is rendered at 672px. LLaVA-1.6 encodes this via high-res tiling, giving it much better OCR than LLaVA-1.5.

In [ ]:
import os
from PIL import Image

print("=" * 60)
print("CONDITION 2: Rendered Image (vision enabled)")
print("=" * 60)

img_predictions, img_correct, img_errors = [], [], []

image_dir = os.path.join(OUTPUT_DIR, "images")

for i, (q, ref) in enumerate(tqdm(zip(questions, references), total=NUM_PROBLEMS, desc="Cond 2")):

    # 🟢 LOAD image instead of generating it
    img_path = os.path.join(image_dir, f"q{i:03d}.png")
    img = Image.open(img_path).convert("RGB")

    # run model
    pred = generate_with_image(img)

    img_predictions.append(pred)
    img_correct.append(answers_match(pred, ref))
    img_errors.append(classify_error(pred, ref))

acc_img = compute_accuracy(img_correct)
print(f"\nCondition 2 Accuracy: {acc_img:.2%}  ({sum(img_correct)}/{NUM_PROBLEMS})")

## 10. Condition 3 — Screenshot (Vision Enabled, Natural Noise) *(Optional)*

Problems are rendered as HTML and screenshotted via headless Chromium, adding font anti-aliasing and layout variation. Set `RUN_SCREENSHOT_CONDITION = True` in §2 to enable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if RUN_SCREENSHOT_CONDITION:
    !apt-get update -qq && apt-get install -y chromium-browser -qq
else:
    print("Condition 3 skipped — skipping Chromium install.")

In [ ]:
scr_predictions, scr_correct, scr_errors = [], [], []
acc_scr = None

if RUN_SCREENSHOT_CONDITION:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    def problem_to_html(question: str) -> str:
        esc = question.replace("<", "&lt;").replace(">", "&gt;")
        return ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
                "<style>body{font-family:Arial,sans-serif;font-size:20px;"
                "max-width:672px;margin:40px auto;padding:20px;"
                "background:#fff;color:#000;line-height:1.6}</style>"
                f"</head><body><p>{esc}</p></body></html>")

    def take_screenshot(html: str, save_path: str) -> Image.Image:
        opts = Options()
        for arg in ["--headless", "--no-sandbox",
                    "--disable-dev-shm-usage", "--window-size=672,600"]:
            opts.add_argument(arg)
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=opts)
        with open("/tmp/vlm_problem.html", "w") as f:
            f.write(html)
        driver.get("file:///tmp/vlm_problem.html")
        driver.save_screenshot(save_path)
        driver.quit()
        return Image.open(save_path)

    print("=" * 60)
    print("CONDITION 3: Screenshot (real-world noise)")
    print("=" * 60)
    for i, (q, ref) in enumerate(tqdm(zip(questions, references), total=NUM_PROBLEMS, desc="Cond 3")):
        path = f"{OUTPUT_DIR}/screenshots/q{i:03d}.png"
        img  = take_screenshot(problem_to_html(q), path)
        pred = generate_with_image(img)
        scr_predictions.append(pred)
        scr_correct.append(answers_match(pred, ref))
        scr_errors.append(classify_error(pred, ref))

    acc_scr = compute_accuracy(scr_correct)
    print(f"\nCondition 3 Accuracy: {acc_scr:.2%}  ({sum(scr_correct)}/{NUM_PROBLEMS})")
else:
    print("Condition 3 skipped. Set RUN_SCREENSHOT_CONDITION = True to enable.")

## 11. Condition 4 — Modality Mismatch (Image ≠ Text Conflict)

Image shows problem *i*, text prompt contains problem *i+1*. We score against both references to determine which modality dominated.

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

print("=" * 60)
print("CONDITION 4: Modality Mismatch (Robust)")
print("=" * 60)

image_dir = os.path.join(OUTPUT_DIR, "images")

mismatch_predictions = []
mismatch_follows = []
mismatch_img_diff = []
mismatch_txt_diff = []

for i in tqdm(range(NUM_PROBLEMS), desc="Condition 4"):

    # ── Load image (problem i)
    img_path = os.path.join(image_dir, f"q{i:03d}.png")
    img = Image.open(img_path).convert("RGB")

    # ── Text is shifted (problem i+1)
    txt_idx = (i + 1) % NUM_PROBLEMS
    txt_question = questions[txt_idx]

    # ── Build prompt (mismatch setup)
    prompt = (
        f"[INST] <image>\n"
        f"Solve the following math problem step by step.\n"
        f"End with '#### <answer>'.\n\n"
        f"Problem: {txt_question} [/INST]"
    )

    inputs = processor(
        text=prompt,
        images=[img],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    n = inputs["input_ids"].shape[1]
    pred = processor.decode(out[0][n:], skip_special_tokens=True).strip()

    # ── Extract numeric answers
    pred_val = extract_numeric_answer(pred)
    img_val = extract_numeric_answer(references[i])
    txt_val = extract_numeric_answer(references[txt_idx])

    # ── Handle failures safely
    if pred_val is None or img_val is None or txt_val is None:
        follows = "invalid"
        diff_img = None
        diff_txt = None
    else:
        diff_img = abs(pred_val - img_val)
        diff_txt = abs(pred_val - txt_val)

        # ── Robust dominance decision
        if diff_img < diff_txt:
            follows = "image"
        elif diff_txt < diff_img:
            follows = "text"
        else:
            follows = "neither"

    mismatch_predictions.append(pred)
    mismatch_follows.append(follows)
    mismatch_img_diff.append(diff_img)
    mismatch_txt_diff.append(diff_txt)


# ── Summary
from collections import Counter

counts = Counter(mismatch_follows)

print(f"\nModality dominance ({NUM_PROBLEMS} samples):")
for k in ["image", "text", "neither", "invalid"]:
    print(f"{k:10s}: {counts.get(k, 0)}")

## 12. Compile Results into a DataFrame

In [ ]:
results = pd.DataFrame({
    "problem_id"       : list(range(NUM_PROBLEMS)),
    "question"         : questions,
    "reference"        : references,

    # ── Condition 1: Text-only
    "pred_text"        : text_predictions,
    "correct_text"     : text_correct,
    "error_text"       : text_errors,

    # ── Condition 2: Rendered Image
    "pred_rendered"    : img_predictions,
    "correct_rendered" : img_correct,
    "error_rendered"   : img_errors,

    # ── Condition 4: Modality Mismatch (UPDATED)
    "mismatch_pred"    : mismatch_predictions,
    "mismatch_follows" : mismatch_follows,
    "mismatch_img_diff": mismatch_img_diff,
    "mismatch_txt_diff": mismatch_txt_diff,
})

# Optional: Screenshot condition
if RUN_SCREENSHOT_CONDITION:
    results["pred_screenshot"]    = scr_predictions
    results["correct_screenshot"] = scr_correct
    results["error_screenshot"]   = scr_errors

# Save
csv_path = f"{OUTPUT_DIR}/gsm8k_llava16_results.csv"
results.to_csv(csv_path, index=False)

print(f"Saved: {csv_path}")
results.head(3)

## 13. Accuracy Summary

In [ ]:
print("=" * 62)
print("   ACCURACY SUMMARY — GSM8K (first 100 problems)")
print("=" * 62)
print(f"  Model : {MODEL_NAME}")
print("-" * 62)

# ── Condition 1 & 2
print(f"  Cond 1 | Text-Only      : {acc_text:.2%}  ({sum(text_correct)}/{NUM_PROBLEMS})")
print(f"  Cond 2 | Rendered Image : {acc_img:.2%}  ({sum(img_correct)}/{NUM_PROBLEMS})")

# ── Condition 3 (optional)
if RUN_SCREENSHOT_CONDITION and 'acc_scr' in globals() and acc_scr is not None:
    print(f"  Cond 3 | Screenshot     : {acc_scr:.2%}  ({sum(scr_correct)}/{NUM_PROBLEMS})")
else:
    print("  Cond 3 | Screenshot     : Not run")

print("=" * 62)

# ── Vision impact
delta = acc_img - acc_text
direction = "HELPS" if delta > 0 else ("HURTS" if delta < 0 else "NEUTRAL")
print(f"\n  Rendered vs Text Δ : {delta:+.2%}  -> Vision {direction} reasoning")

# ── Condition 4: Modality dominance (robust version)
if 'follow_counts' in globals():
    ic = counts.get("image", 0)
    tc = counts.get("text", 0)
    nc = counts.get("neither", 0)
    iv = counts.get("invalid", 0)

    total_valid = ic + tc + nc
    img_pct = ic / total_valid if total_valid > 0 else 0.0
    txt_pct = tc / total_valid if total_valid > 0 else 0.0
    nei_pct = nc / total_valid if total_valid > 0 else 0.0

    if ic > tc:   dom = f"IMAGE ({ic} vs {tc})"
    elif tc > ic: dom = f"TEXT ({tc} vs {ic})"
    else:         dom = f"TIE ({ic} each)"

    print("\n  Condition 4 — Modality Mismatch:")
    print(f"    Image   : {ic} ({img_pct:.2%})")
    print(f"    Text    : {tc} ({txt_pct:.2%})")
    print(f"    Neither : {nc} ({nei_pct:.2%})")
    if iv > 0:
        print(f"    Invalid : {iv}")
    print(f"\n  Mismatch dominant  : {dom}")
else:
    print("\n  Cond 4 | Mismatch       : Not run yet")

## 14. Accuracy Bar Chart

In [ ]:
conditions = ["Cond 1\nText-Only", "Cond 2\nRendered"]
accuracies = [acc_text, acc_img]
colors     = ["#4C72B0", "#55A868"]

if RUN_SCREENSHOT_CONDITION and acc_scr is not None:
    conditions.append("Cond 3\nScreenshot")
    accuracies.append(acc_scr)
    colors.append("#C44E52")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(conditions, [a * 100 for a in accuracies],
              color=colors, width=0.5, edgecolor="black")
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{acc:.1%}", ha="center", va="bottom", fontsize=13, fontweight="bold")

ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title(f"GSM8K Zero-Shot Accuracy by Modality\n{MODEL_NAME}", fontsize=11)
ax.axhline(acc_text * 100, color="#4C72B0", linestyle="--", alpha=0.5, label="Text baseline")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/accuracy_by_condition.png", dpi=150)
plt.show()

## 15. Error Type Breakdown

In [ ]:
error_categories = ["correct", "arithmetic_error", "reasoning_error", "no_number", "vision_error"]

cond_labels = ["Text-Only", "Rendered"]
cond_errors = [text_errors, img_errors]
if RUN_SCREENSHOT_CONDITION:
    cond_labels.append("Screenshot")
    cond_errors.append(scr_errors)

x, n = np.arange(len(error_categories)), len(cond_labels)
bar_w, palette = 0.8 / n, ["#4C72B0", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(10, 5))
for idx, (label, errors) in enumerate(zip(cond_labels, cond_errors)):
    c = Counter(errors)
    counts = [c.get(cat, 0) for cat in error_categories]
    offset = (idx - n / 2 + 0.5) * bar_w
    bars = ax.bar(x + offset, counts, bar_w, label=label,
                  color=palette[idx], edgecolor="black", alpha=0.85)
    for b, v in zip(bars, counts):
        if v > 0:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
                    str(v), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", "\n") for c in error_categories], fontsize=11)
ax.set_ylabel("Number of Problems", fontsize=12)
ax.set_title("Error Type Breakdown by Condition", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/error_breakdown.png", dpi=150)
plt.show()

## 16. Modality Mismatch Chart & Interpretation

In [ ]:
from collections import Counter

mismatch_df = pd.DataFrame({
    "problem_id"      : list(range(NUM_PROBLEMS)),
    "image_question"  : questions,
    "image_reference" : references,
    "text_question"   : [questions[(i+1) % NUM_PROBLEMS] for i in range(NUM_PROBLEMS)],
    "text_reference"  : [references[(i+1) % NUM_PROBLEMS] for i in range(NUM_PROBLEMS)],

    "prediction"      : mismatch_predictions,
    "follows"         : mismatch_follows,   # image / text / equal / invalid

    "img_diff"        : mismatch_img_diff,
    "txt_diff"        : mismatch_txt_diff,
})

mismatch_df.to_csv(f"{OUTPUT_DIR}/mismatch_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/mismatch_results.csv")


# ── IMPORTANT: build Counter properly
follow_counts = Counter(mismatch_follows)


# ── Plot modality dominance
labels = ["Follows Image", "Follows Text", "Equal", "Invalid"]
keys   = ["image", "text", "equal", "invalid"]

counts = [follow_counts.get(k, 0) for k in keys]

colors = ["#55A868", "#4C72B0", "#8172B2", "#999999"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, counts, color=colors, edgecolor="black", width=0.5)

for bar, v in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        str(v),
        ha="center",
        va="bottom",
        fontsize=13,
        fontweight="bold"
    )

ax.set_ylabel("Number of Problems", fontsize=12)
ax.set_title(f"Mismatch: Which modality does the model follow?\n{MODEL_NAME}", fontsize=11)
ax.set_ylim(0, NUM_PROBLEMS + 10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mismatch_dominance.png", dpi=150)
plt.show()


# ── Interpretation
img_c = follow_counts.get('image', 0)
txt_c = follow_counts.get('text', 0)

if img_c > txt_c:
    print(f"{MODEL_NAME} PREFERS IMAGE ({img_c} vs {txt_c})")
elif txt_c > img_c:
    print(f"{MODEL_NAME} PREFERS TEXT ({txt_c} vs {img_c})")
else:
    print(f"{MODEL_NAME} shows NO CLEAR PREFERENCE ({img_c} vs {txt_c})")

## 17. Disagreements Between Conditions 1 & 2

In [ ]:
disagreements = results[results["correct_text"] != results["correct_rendered"]][
    ["problem_id", "question", "reference",
     "pred_text", "correct_text", "error_text",
     "pred_rendered", "correct_rendered", "error_rendered"]
]
print(f"Disagreements between Conditions 1 & 2: {len(disagreements)}")
print(f"  Text correct, Image wrong : {disagreements['correct_text'].sum()}")
print(f"  Image correct, Text wrong : {disagreements['correct_rendered'].sum()}")
disagreements.to_csv(f"{OUTPUT_DIR}/disagreements.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/disagreements.csv")
disagreements.head(5)

## 18. Interpretation & Next Steps

In [ ]:
print("=" * 62)
print("INTERPRETATION — LLaVA-1.6 Mistral-7B")
print("=" * 62)
delta = acc_img - acc_text
if abs(delta) < 0.02:
    verdict = f"Vision is NEUTRAL (delta={delta:+.2%}). Both modalities perform comparably."
elif delta > 0:
    verdict = (f"Vision HELPS by {delta:+.2%}. High-res tiling may aid number/structure parsing.")
else:
    verdict = (f"Vision HURTS by {delta:+.2%}. Image encoding adds noise to Mistral reasoning.")

print(f"\nText-Only : {acc_text:.2%}")
print(f"Rendered  : {acc_img:.2%}")
if RUN_SCREENSHOT_CONDITION and acc_scr is not None:
    print(f"Screenshot: {acc_scr:.2%}  (delta vs text: {acc_scr - acc_text:+.2%})")
print(f"\nVerdict   : {verdict}")

for label, errors in zip(cond_labels, cond_errors):
    wrong = [e for e in errors if e != "correct"]
    if wrong:
        top = Counter(wrong).most_common(1)[0]
        print(f"  [{label}] top error: '{top[0]}' ({top[1]} cases)")

ic, tc = follow_counts.get('image', 0), follow_counts.get('text', 0)
print(f"\nMismatch  : image={ic}, text={tc}, neither={follow_counts.get('neither',0)}")
print(f"            dominant = {'IMAGE' if ic > tc else 'TEXT' if tc > ic else 'TIE'}")
print(f"\nOutputs saved to: {OUTPUT_DIR}/")

## 19. Export & Save *(Optional)*

In [ ]:
!zip -r vlm_benchmark_outputs.zip {OUTPUT_DIR}/
print("Zipped: vlm_benchmark_outputs.zip")

In [ ]:
# Uncomment to save to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp vlm_benchmark_outputs.zip /content/drive/MyDrive/
# print("Copied to Google Drive.")